In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 处理 Kaggle 上 P100 + 新版 PyTorch 不兼容的问题
# 3. 导入后续会用到的基础库
# 4. 固定随机种子，方便复现实验
# 5. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


def choose_device():
    # 选择当前 Kaggle 环境里真正能安全使用的设备。
    if not torch.cuda.is_available():
        print('GPU not found, using CPU. 也能跑小实验，但会慢一些。')
        return 'cpu'

    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print('GPU:', name)
    print('CUDA capability:', f'sm_{props.major}{props.minor}')

    # Kaggle 有时会分配 Tesla P100。P100 是 sm_60。
    # 如果当前 PyTorch 构建不支持 sm_60，torch.cuda.is_available() 仍可能是 True，
    # 但真正执行 CUDA kernel 时会失败，所以这里主动退回 CPU。
    if props.major < 7:
        print()
        print('注意：当前 GPU 是 P100/sm_60，但这个 PyTorch 构建可能不支持 sm_60。')
        print('本 notebook 将临时使用 CPU。想用 GPU，请在 Kaggle Accelerator 里优先选择 T4。')
        print('如果 Kaggle 只能给 P100，也可以改装兼容旧架构的 PyTorch，但初学阶段先不折腾环境。')
        print()
        return 'cpu'

    return 'cuda'


device = choose_device()


# 固定随机种子。
# 初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device == 'cuda':
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对学习实验来说，可以让矩阵乘法更快。
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print('Using device:', device)
print('Seed:', seed)


# 后续我们会按这个顺序继续写：
# Cell 2：按官方 GPT-2 复现路线准备 OpenWebText，并用 GPT-2 BPE 变成 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：实现最小语言模型，先理解训练闭环
# Cell 5：实现单头 causal self-attention
# Cell 6：实现多头注意力
# Cell 7：实现 MLP 和 Block
# Cell 8：组装 GPTConfig 和 GPT
# Cell 9：写训练循环
# Cell 10：写 generate 生成文本


In [ ]:
# 第 2 个 cell：官方 OpenWebText + GPT-2 BPE 数据准备
#
# 这一格对应 nanoGPT 官方源码：
#   nanoGPT-reference/data/openwebtext/prepare.py
#
# 你提到的“基于 openwebtext dataset 复现 GPT-2”，对应的就是这条路线：
#
#   OpenWebText -> GPT-2 BPE tokenizer -> train.bin / val.bin
#
# 注意：完整 OpenWebText 很大。nanoGPT 官方注释里写到：
#   Hugging Face cache 约 54GB
#   train.bin 约 17GB
#   val.bin 约 8.5MB
#   train 约 9B tokens
#
# 所以这一格优先使用 Kaggle Input 里已经准备好的 train.bin / val.bin。
# 如果你确认 Kaggle 磁盘和时间足够，再把 PREPARE_OPENWEBTEXT_FROM_HF 改成 True，
# 它会按官方 prepare.py 的方式从 Hugging Face 下载并预处理完整 OpenWebText。

import os
import subprocess
import sys
from pathlib import Path

import numpy as np


PREPARE_OPENWEBTEXT_FROM_HF = False


def ensure_package(import_name, pip_name=None):
    # 缺少依赖时自动安装，方便在 Kaggle 新环境中直接运行。
    try:
        return __import__(import_name)
    except ModuleNotFoundError:
        package_name = pip_name or import_name
        print(f'当前环境没有 {package_name}，正在安装。')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        return __import__(import_name)


tiktoken = ensure_package('tiktoken')
ensure_package('tqdm')
from tqdm import tqdm


# GPT-2 BPE tokenizer。
# BPE 可以理解成“常见文本片段优先合并”的分词方式。
# GPT-2 的普通词表大小是 50257，其中结束符通常是 50256。
# nanoGPT 从零训练 GPT-2 规模模型时，默认把 vocab_size padding 到 50304，
# 这样矩阵维度更适合 GPU 计算。
enc = tiktoken.get_encoding('gpt2')
gpt2_vocab_size = enc.n_vocab
vocab_size = 50304

print('Tokenizer:', enc.name)
print('GPT-2 原始词表大小:', gpt2_vocab_size)
print('GPT-2 eot_token:', enc.eot_token)
print('nanoGPT 从零训练默认 vocab_size:', vocab_size)


example_text = 'The quick brown fox jumps over the lazy dog.'
example_ids = enc.encode_ordinary(example_text)

print()
print('BPE 编码/解码小例子：')
print('原始文本:', example_text)
print('编码后 token id:', example_ids)
print('解码回去:', enc.decode(example_ids))
print('每个 token id 对应的文本片段：')
for token_id in example_ids:
    print(f'{token_id:>6} -> {enc.decode([token_id])!r}')


def find_prepared_openwebtext_bins():
    # 优先在 Kaggle Input 里找已经准备好的 OpenWebText 二进制 token 文件。
    # 这样不占 /kaggle/working 的 Output 空间。
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists():
        return None

    train_candidates = sorted(kaggle_input.rglob('train.bin'))
    val_candidates = sorted(kaggle_input.rglob('val.bin'))

    for train_path in train_candidates:
        same_dir_val = train_path.with_name('val.bin')
        if same_dir_val.exists():
            return train_path, same_dir_val

    if train_candidates and val_candidates:
        return train_candidates[0], val_candidates[0]

    return None


def prepare_openwebtext_from_hf():
    # 这部分严格对应 nanoGPT 官方 data/openwebtext/prepare.py。
    # 完整运行会下载和处理大量数据，Kaggle 免费环境可能因为空间或时间失败。
    ensure_package('datasets')
    from datasets import load_dataset

    if Path('/kaggle/working').exists():
        data_dir = Path('/kaggle/working/data/openwebtext')
    else:
        data_dir = Path('data/openwebtext')
    data_dir.mkdir(parents=True, exist_ok=True)

    num_proc = min(8, max(1, (os.cpu_count() or 2) // 2))
    num_proc_load_dataset = num_proc

    print('开始加载 OpenWebText。第一次运行会下载大量数据。')
    dataset = load_dataset('openwebtext', num_proc=num_proc_load_dataset)

    split_dataset = dataset['train'].train_test_split(
        test_size=0.0005,
        seed=2357,
        shuffle=True,
    )
    split_dataset['val'] = split_dataset.pop('test')
    print(split_dataset)

    def process(example):
        ids = enc.encode_ordinary(example['text'])
        ids.append(enc.eot_token)
        return {'ids': ids, 'len': len(ids)}

    print('开始 GPT-2 BPE 分词。')
    tokenized = split_dataset.map(
        process,
        remove_columns=['text'],
        desc='tokenizing the splits',
        num_proc=num_proc,
    )

    bin_paths = {}
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = data_dir / f'{split}.bin'
        arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
        total_batches = 1024
        idx = 0

        print()
        print(f'开始写入 {filename}')
        print(f'{split} token 总数:', f'{int(arr_len):,}')

        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename.name}'):
            batch = dset.shard(
                num_shards=total_batches,
                index=batch_idx,
                contiguous=True,
            ).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        arr.flush()
        bin_paths[split] = filename

    return bin_paths['train'], bin_paths['val']


prepared_bins = find_prepared_openwebtext_bins()

if prepared_bins is not None:
    train_bin_path, val_bin_path = prepared_bins
    print()
    print('发现 Kaggle Input 中已有 OpenWebText bin 文件，直接读取：')
    print('train.bin:', train_bin_path)
    print('val.bin:', val_bin_path)
elif PREPARE_OPENWEBTEXT_FROM_HF:
    train_bin_path, val_bin_path = prepare_openwebtext_from_hf()
else:
    raise FileNotFoundError(
        '没有在 /kaggle/input 中找到 train.bin / val.bin。\n'
        '推荐先在 Kaggle Add Input 里添加已经准备好的 OpenWebText 数据集。\n'
        '如果你确认空间和时间足够，也可以把 PREPARE_OPENWEBTEXT_FROM_HF 改成 True，按官方脚本现场下载预处理。'
    )


# 后面训练时不需要把完整 train.bin 读进内存。
# np.memmap 会建立“文件视图”，用到哪一段再读哪一段。
train_data = np.memmap(train_bin_path, dtype=np.uint16, mode='r')
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode='r')

print()
print('OpenWebText 数据准备完成。')
print('train.bin:', train_bin_path, 'tokens:', f'{len(train_data):,}')
print('val.bin:', val_bin_path, 'tokens:', f'{len(val_data):,}')
print('训练集前 40 个 token id:')
print(train_data[:40].tolist())
print('解码前 40 个 token:')
print(enc.decode(train_data[:40].tolist()))


def encode(s):
    # 把字符串变成 GPT-2 BPE token id 列表。
    return enc.encode_ordinary(s)


def decode(ids):
    # 把 GPT-2 BPE token id 列表还原成字符串。
    return enc.decode(ids)


# 这一格产生的关键变量：
#   enc / encode / decode       GPT-2 BPE 分词器和辅助函数
#   vocab_size                  nanoGPT 从零训练 GPT-2 路线默认词表大小，50304
#   train_bin_path / val_bin_path
#   train_data / val_data       np.memmap 形式的 OpenWebText token 序列
#
# 下一格 Cell 3 会讲：
#   如何从 train_data 中随机取一批连续 token，
#   构造 x 和 y，让模型学习“看到前文，预测下一个 token”。
